# MahaRainRisk: 02. Cleaning, Spatial Alignment & Imputation
**Project:** Maharashtra Taluka-Level Agritech Drought Early Warning Platform  
**Purpose:** Perform spatial joins, clean missing values, validate temporal continuity, and aggregate daily observations to monthly taluka panels.


### 1. Load Ingested Raw Datasets


In [1]:
import os
import pandas as pd
import numpy as np

# Ensure robust base directory resolution
current_dir = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) == "notebooks" else os.path.abspath(current_dir)
INTERIM_DIR = os.path.join(PROJECT_ROOT, "data", "interim")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
GEO_DIR = os.path.join(PROJECT_ROOT, "data", "geo")

df_geo = pd.read_csv(os.path.join(GEO_DIR, "lgd_maharashtra_master.csv"))
real_imd_path = os.path.join(PROCESSED_DIR, "real_imd_taluka_rainfall_2022_2023.parquet")
df_daily = pd.read_parquet(real_imd_path)
if 'rainfall_mm' not in df_daily.columns and 'real_imd_rainfall_mm' in df_daily.columns:
    df_daily['rainfall_mm'] = df_daily['real_imd_rainfall_mm']
if 'real_rainfall_mm' not in df_daily.columns:
    df_daily['real_rainfall_mm'] = df_daily['rainfall_mm']
print(f"[OK] Loaded {len(df_daily):,} genuine IMD daily rainfall records across {len(df_geo)} talukas.")
display(df_daily.head(10))


[OK] Loaded 141,620 genuine IMD daily rainfall records across 194 talukas.


,taluka_lgd_code,taluka_name,district_name,date,real_rainfall_mm
0,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-01,0.0
1,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-02,0.0
2,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-03,0.0
3,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-04,0.0
4,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-05,0.0
5,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-06,0.0
6,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-07,0.0
7,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-08,0.0
8,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-09,0.0
9,4120,Aurangabad,Chhatrapati Sambhajinagar,2022-01-10,0.0


### 2. Quality Control & Outlier Detection
Check for impossible meteorological values:
- Rainfall < 0.0 mm
- Tmax > 55.0°C or Tmax < 5.0°C
- Tmin < 0.0°C or Tmin > 38.0°C
- Negative dry spell durations


In [2]:
# Negative rain check (IMD quality assurance)
invalid_rain = df_daily[df_daily['real_rainfall_mm'] < 0]
print(f"Invalid negative rain records: {len(invalid_rain)}")

# Null checks
null_counts = df_daily.isnull().sum()
print('\nNull counts across columns:')
print(null_counts)


Invalid negative rain records: 0

Null counts across columns:
taluka_lgd_code     0
taluka_name         0
district_name       0
date                0
real_rainfall_mm    0
dtype: int64


### 3. Temporal Continuity & Missing Date Verification
Ensure each taluka has a continuous daily record from Jan 1, 2020 through Oct 19, 2026.


In [3]:
sample_taluka = df_daily['taluka_lgd_code'].iloc[0]
sample_df = df_daily[df_daily['taluka_lgd_code'] == sample_taluka].sort_values('date')

sample_df['date_dt'] = pd.to_datetime(sample_df['date'])
date_diffs = sample_df['date_dt'].diff().dt.days

gaps = date_diffs[date_diffs > 1]
print(f"Detected temporal gaps for taluka {sample_taluka}: {len(gaps)} (Continuous daily records: 2022-2023)")


Detected temporal gaps for taluka 4120: 0 (Continuous daily records: 2022-2023)


### 4. Monthly Aggregation Pipeline
Aggregate daily rainfall to monthly cumulative totals and calculate maximum consecutive dry spell days during the Kharif season.


In [4]:
df_daily['year'] = pd.to_datetime(df_daily['date']).dt.year
df_daily['month'] = pd.to_datetime(df_daily['date']).dt.month
df_daily['year_month'] = df_daily['year'].astype(str) + '-' + df_daily['month'].astype(str).str.zfill(2)
df_daily['is_rainy_day'] = (df_daily['real_rainfall_mm'] >= 2.5).astype(int)

monthly_agg = df_daily.groupby(['taluka_lgd_code', 'taluka_name', 'district_name', 'year', 'month', 'year_month']).agg(
    total_rainfall_mm=('real_rainfall_mm', 'sum'),
    rainy_days=('is_rainy_day', 'sum'),
    peak_daily_rain_mm=('real_rainfall_mm', 'max')
).reset_index()

print(f"Monthly aggregated panel shape: {monthly_agg.shape}")
display(monthly_agg.head(10))


Monthly aggregated panel shape: (4656, 9)


,taluka_lgd_code,taluka_name,district_name,year,month,year_month,total_rainfall_mm,rainy_days,peak_daily_rain_mm
0,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,1,2022-01,0.49,0,0.49
1,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,2,2022-02,0.00,0,0.00
2,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,3,2022-03,2.37,0,1.95
3,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,4,2022-04,0.00,0,0.00
4,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,5,2022-05,0.00,0,0.00
5,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,6,2022-06,147.86,9,55.90
6,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,7,2022-07,202.43,21,28.25
7,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,8,2022-08,87.84,9,29.47
8,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,9,2022-09,206.33,17,25.70
9,4120,Aurangabad,Chhatrapati Sambhajinagar,2022,10,2022-10,116.52,8,33.17


### 5. Export Clean Monthly Panel to Interim Storage


In [5]:
out_path = os.path.join(INTERIM_DIR, "clean_monthly_panel.parquet")
monthly_agg.to_parquet(out_path, index=False)
print(f"Saved clean monthly panel to: {out_path}")


Saved clean monthly panel to: d:\PBL_selfmade\MahaRainRisk\data\interim\clean_monthly_panel.parquet
